# Imidazolone & Thiazolone Hit Prioritization

Phase 2 prioritization for generated COX-2 inhibitor candidates derived from
Phase 1 product sets.

**Pipeline:** ```Generated products → QED scoring → Bioavailability filter (4/5 rules)```  

**Series handling:** Imidazolones and thiazolones are processed as two
independent datasets.  

**Visualization:** QED histograms are generated for accepted and rejected
sets using the combined distributions of both series.

**Clustering input control:** accepted compounds are re-filtered by
independent per-series price controls (max price, acceptance rate, max N)
before export to `mol_files/7. Clustering/`.

In [ ]:
from __future__ import annotations

import py_utils.phase2_hit_prioritization as p2

FORCE_RECOMPUTE = False
INPUT_FILTER_MODE = "brenkpains"

# Price controls for clustering inputs (None disables each control)
MAX_PRICE_IMIDAZOLONES = None
MAX_PRICE_THIAZOLONES = None
ACCEPTANCE_RATE_IMIDAZOLONES = 0.67
ACCEPTANCE_RATE_THIAZOLONES = 0.67
MAX_SAMPLE_SIZE_IMIDAZOLONES = None
MAX_SAMPLE_SIZE_THIAZOLONES = None

print("phase2_hit_prioritization module loaded")
print(f"Input filter mode: {INPUT_FILTER_MODE}")
print(f"Force recompute QED cache: {FORCE_RECOMPUTE}")
print("Price control order: max_price -> acceptance_rate -> max_sample_size")

## 📥 1. Load Generated Products

Load the latest counted CSV files from Phase 1 (Brenk+PAINS outputs) for
both product families.

In [ ]:
df_imidazolones_generated, df_thiazolones_generated, imidazolones_source, thiazolones_source = p2.load_generated_product_sets(
    filter_mode=INPUT_FILTER_MODE,
)

p2.report_df_size(df_imidazolones_generated, "Imidazolones - Generated")
p2.report_df_size(df_thiazolones_generated, "Thiazolones - Generated")

## 🔸 2. QED Scoring

Compute Quantitative Estimate of Drug-likeness (QED) and insert it between
`PriceMol` and `tPSA`.

> Cache is stored in `mol_files/6. QED/.cache/` for restart-safe reuse.

In [ ]:
df_imidazolones_qed, imidazolones_qed_cache = p2.load_or_compute_qed(
    df_imidazolones_generated,
    stage_name="Imidazolones",
    force_recompute=FORCE_RECOMPUTE,
)
df_thiazolones_qed, thiazolones_qed_cache = p2.load_or_compute_qed(
    df_thiazolones_generated,
    stage_name="Thiazolones",
    force_recompute=FORCE_RECOMPUTE,
)

p2.report_df_size(df_imidazolones_qed, "Imidazolones - QED")
p2.report_df_size(df_thiazolones_qed, "Thiazolones - QED")

## 🔸 3. Bioavailability Filter (4/5 Rule Set)

Apply Lipinski, Ghose, Egan, Muegge, and Veber criteria and keep compounds
with at most one violated rule.

`Violation` is inserted immediately after `QED` and stores violated rule
names (`none` when no rule is violated).

In [ ]:
df_imidazolones_druglike, df_imidazolones_nondruglike = p2.filter_bioavailability(df_imidazolones_qed)
df_thiazolones_druglike, df_thiazolones_nondruglike = p2.filter_bioavailability(df_thiazolones_qed)

p2.report_df_size(df_imidazolones_druglike, "Imidazolones - Druglike")
p2.report_df_size(df_thiazolones_druglike, "Thiazolones - Druglike")
p2.report_df_size(df_imidazolones_nondruglike, "Imidazolones - NonDruglike")
p2.report_df_size(df_thiazolones_nondruglike, "Thiazolones - NonDruglike")

## 📊 4. QED Distribution

Plot QED histograms for:
- Rejected compounds (imidazolones + thiazolones)
- Accepted compounds (imidazolones + thiazolones)

In [ ]:
_fig, _axes = p2.plot_qed_histograms(
    df_imidazolones_druglike=df_imidazolones_druglike,
    df_thiazolones_druglike=df_thiazolones_druglike,
    df_imidazolones_nondruglike=df_imidazolones_nondruglike,
    df_thiazolones_nondruglike=df_thiazolones_nondruglike,
)

## 📤 5. Export (Bioavailability Outputs)

Save accepted and rejected sets in `mol_files/6. QED/` using row-count
suffix naming conventions.

In [ ]:
bioavailability_paths = p2.save_bioavailability_outputs(
    df_imidazolones_druglike=df_imidazolones_druglike,
    df_thiazolones_druglike=df_thiazolones_druglike,
    df_imidazolones_nondruglike=df_imidazolones_nondruglike,
    df_thiazolones_nondruglike=df_thiazolones_nondruglike,
)

## 🔸 6. Price Re-Filtering (Clustering Input)

Apply independent per-series price controls in this exact order:
1. `max_price`
2. `acceptance_rate` (fraction 0-1, keeps floor(rate*N), min 1 when rate > 0)
3. `max_sample_size` (top-N cheapest after previous controls)

In [ ]:
df_imidazolones_input, df_imidazolones_rejected_pricectrl = p2.apply_price_controls(
    df_imidazolones_druglike,
    max_price=MAX_PRICE_IMIDAZOLONES,
    acceptance_rate=ACCEPTANCE_RATE_IMIDAZOLONES,
    max_sample_size=MAX_SAMPLE_SIZE_IMIDAZOLONES,
)

df_thiazolones_input, df_thiazolones_rejected_pricectrl = p2.apply_price_controls(
    df_thiazolones_druglike,
    max_price=MAX_PRICE_THIAZOLONES,
    acceptance_rate=ACCEPTANCE_RATE_THIAZOLONES,
    max_sample_size=MAX_SAMPLE_SIZE_THIAZOLONES,
)

p2.report_df_size(df_imidazolones_input, "Imidazolones - Clustering Input")
p2.report_df_size(df_thiazolones_input, "Thiazolones - Clustering Input")
p2.report_df_size(df_imidazolones_rejected_pricectrl, "Imidazolones - Rejected PriceCtrl")
p2.report_df_size(df_thiazolones_rejected_pricectrl, "Thiazolones - Rejected PriceCtrl")

## 📤 7. Export (Clustering Inputs)

Save accepted clustering inputs and price-control rejections in
`mol_files/7. Clustering/` with row-count suffix naming conventions.

In [ ]:
clustering_paths = p2.save_price_control_outputs(
    df_imidazolones_input=df_imidazolones_input,
    df_thiazolones_input=df_thiazolones_input,
    df_imidazolones_rejected_pricectrl=df_imidazolones_rejected_pricectrl,
    df_thiazolones_rejected_pricectrl=df_thiazolones_rejected_pricectrl,
)

print("\n=== Hit Prioritization Complete ===")
print(f"Imidazolones clustering input: {len(df_imidazolones_input):,} compounds")
print(f"Thiazolones clustering input:  {len(df_thiazolones_input):,} compounds")
print(f"Imidazolones rejected by price control: {len(df_imidazolones_rejected_pricectrl):,} compounds")
print(f"Thiazolones rejected by price control:  {len(df_thiazolones_rejected_pricectrl):,} compounds")